# Data Setup — Heterogeneous Ingestion Lab

Creates three files used by the main lab:
- `orders.csv`
- `access.log`
- `users_fallback.json`

> Safe to re-run.


In [20]:
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
from datetime import datetime, timedelta

DATA_DIR = Path('./data'); DATA_DIR.mkdir(exist_ok=True)
print('Data directory:', DATA_DIR.resolve())

Data directory: /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data


## 1) `orders.csv`

In [13]:
n_orders = 10_000
n_users = 1_000
n_logs = 5_000

In [27]:
rng = np.random.default_rng(42)
user_ids = rng.integers(1, n_users, size=n_orders)
order_totals = np.round(rng.uniform(10, 200, size=n_orders), 2)
start = datetime(2025, 8, 15, 9, 0, 0)
timestamps = [start + timedelta(minutes=int(x)) for x in rng.integers(0, 7*24*60, size=n_orders)]

df_orders = pd.DataFrame({
    'order_id': [f'O{i}' for i in range(1, n_orders+1)],
    'user_id': user_ids,
    'order_total': order_totals,
    'order_ts': [t.isoformat() for t in timestamps]
})
df_orders.loc[5, 'order_id'] = df_orders.loc[0, 'order_id']
display(df_orders.head()); print('Rows:', len(df_orders), 'Unique order_ids:', df_orders['order_id'].nunique())

,order_id,user_id,order_total,order_ts
0,O1,90,51.45,2025-08-16T23:55:00
1,O2,774,135.96,2025-08-17T19:23:00
2,O3,654,155.70,2025-08-21T09:37:00
3,O4,439,41.66,2025-08-16T02:16:00
4,O5,433,18.33,2025-08-16T03:14:00


Rows: 10000 Unique order_ids: 9999


In [28]:
out = Path(os.path.join(DATA_DIR, 'orders.csv'))
df_orders.to_csv(out, index=False); print('Wrote:', out.resolve())

Wrote: /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/orders.csv


## 2) `access.log`

In [16]:
lines = []
base_time = datetime(2025, 8, 22, 10, 0, 0)
paths = ['/', '/home', '/product/42', '/product/7', '/cart', '/checkout']
rng = np.random.default_rng(7)
for i in range(n_logs):
    ts = base_time + timedelta(seconds=int(rng.integers(0, 6*3600)))
    uid = int(rng.integers(1, n_users))
    path = paths[int(rng.integers(0, len(paths)))]
    status = int(rng.choice([200, 200, 200, 302, 404]))
    bytes_sent = int(rng.integers(350, 4000))
    ua = 'Mozilla/5.0'
    line = f'127.0.0.1 - - [{ts.isoformat()}] "GET {path}?uid={uid} HTTP/1.1" {status} {bytes_sent} "-" "{ua}"'
    lines.append(line)
print('\n'.join(lines[:3]))

127.0.0.1 - - [2025-08-22T15:40:09] "GET /cart?uid=625 HTTP/1.1" 404 2460 "-" "Mozilla/5.0"
127.0.0.1 - - [2025-08-22T14:39:14] "GET /home?uid=833 HTTP/1.1" 200 1445 "-" "Mozilla/5.0"
127.0.0.1 - - [2025-08-22T11:42:37] "GET /checkout?uid=873 HTTP/1.1" 200 2174 "-" "Mozilla/5.0"


In [23]:
Path(os.path.join(DATA_DIR, 'access.log')).write_text('\n'.join(lines))
print('Wrote:', Path(os.path.join(DATA_DIR, 'access.log')).resolve())

Wrote: /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/access.log


## 3) `users_fallback.json`

In [24]:
fallback_users = [
    {'id':1,'name':'Leanne Graham','username':'Bret','email':'leanne@example.com','address':{'city':'Gwenborough'}},
    {'id':2,'name':'Ervin Howell','username':'Antonette','email':'ervin@example.com','address':{'city':'Wisokyburgh'}},
    {'id':3,'name':'Clementine Bauch','username':'Samantha','email':'clementine@example.com','address':{'city':'McKenzie'}},
    {'id':4,'name':'Patricia Lebsack','username':'Karianne','email':'patricia@example.com','address':{'city':'South Elvis'}},
    {'id':5,'name':'Chelsey Dietrich','username':'Kamren','email':'chelsey@example.com','address':{'city':'Roscoe'}}
]
path = Path(os.path.join(DATA_DIR, 'users_fallback.json'))
path.write_text(json.dumps(fallback_users, indent=2))
print('Wrote:', path.resolve())

Wrote: /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/users_fallback.json


## 4) Verify

In [25]:
for p in ['orders.csv','access.log','users_fallback.json']:
    fp = Path(os.path.join(DATA_DIR, p))
    print(('[OK]' if fp.exists() else '[MISSING]'), fp.resolve())

[OK] /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/orders.csv
[OK] /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/access.log
[OK] /home/jupyter-samread/L5 DE/F. Data Collection and Ingestion/Topic 2/data/users_fallback.json
